# Ground Truth Builder

This notebook builds a structured ground truth document for evaluating the AI lease analysis agent.

**Sources used:**
1. Reference PDFs already in `rag_data/info/`
2. Scraped web content from public tenant-rights resources

**Output:** `ground_truth/ground_truth.json` — a structured list of clause types with expected labels and severity ranges.

Run this notebook once before running `02_evaluate_output.ipynb`.

## Cell 1: Setup

In [1]:
import os
import json
import re
from pathlib import Path
from datetime import date

import requests
from bs4 import BeautifulSoup
import pypdf
from openai import OpenAI

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR     = Path("..").resolve()
RAG_INFO_DIR = BASE_DIR / "rag_data" / "info"
SECRETS_PATH = BASE_DIR.parent / "secrets.txt"   # ../secrets.txt relative to evaluation/
OUTPUT_DIR   = Path("ground_truth")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Load API key ────────────────────────────────────────────────────────────
def load_secrets(path=SECRETS_PATH):
    secrets = {}
    try:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    k, v = line.split("=", 1)
                    secrets[k.strip()] = v.strip()
    except FileNotFoundError:
        print(f"secrets.txt not found at {path}. Falling back to environment variable.")
    return secrets

secrets = load_secrets()
api_key = secrets.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Add it to secrets.txt or set it as an environment variable.")

client = OpenAI(api_key=api_key)
print("Setup complete.")
print(f"RAG info dir: {RAG_INFO_DIR}")
print(f"Output dir:   {OUTPUT_DIR.resolve()}")

Setup complete.
RAG info dir: C:\Users\fblel\OneDrive\Desktop\Courses\Winter 2026\AI Financial Info\Files\busn30135_labs\aiforfinancialinformation\rag_data\info
Output dir:   C:\Users\fblel\OneDrive\Desktop\Courses\Winter 2026\AI Financial Info\Files\busn30135_labs\aiforfinancialinformation\evaluation\ground_truth


## Cell 2: Load Reference PDFs

Extract text from the two reference documents already in `rag_data/info/`.

In [2]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract all text from a PDF file."""
    reader = pypdf.PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)

pdf_texts = {}
pdf_files = list(RAG_INFO_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"WARNING: No PDFs found in {RAG_INFO_DIR}")
else:
    for pdf_path in pdf_files:
        print(f"Loading: {pdf_path.name}")
        text = extract_text_from_pdf(pdf_path)
        pdf_texts[pdf_path.name] = text
        print(f"  -> {len(text):,} characters, {len(reader := pypdf.PdfReader(str(pdf_path))).pages if False else len(pypdf.PdfReader(str(pdf_path)).pages)} pages")

print(f"\nLoaded {len(pdf_texts)} PDF(s).")

Loading: 10-deadly-sins-of-a-lease.pdf
  -> 3,522 characters, 1 pages
Loading: How to Identify and Fix Illegal Lease Clauses.pdf


  -> 8,454 characters, 5 pages

Loaded 2 PDF(s).


## Cell 3: Web Scraping

Scrape public tenant-rights resources to supplement the PDF content.
Failures are handled gracefully — the notebook will continue even if a site is unavailable.

In [3]:
WEB_SOURCES = [
    {
        "name": "NOLO – Illegal Lease and Rental Clauses",
        "url": "https://www.nolo.com/legal-encyclopedia/illegal-lease-rental-clauses.html",
    },
    {
        "name": "HUD – Tenant Rights",
        "url": "https://www.hud.gov/topics/rental_assistance/tenantrights",
    },
    {
        "name": "Illinois Attorney General – Landlord Tenant Rights",
        "url": "https://illinoisattorneygeneral.gov/rights/landlordtenants.html",
    },
]

def scrape_url(url: str, timeout: int = 15) -> str:
    """Scrape visible text from a URL. Returns empty string on failure."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (compatible; academic-research-bot/1.0; "
            "+https://github.com/almostgraduatedmba/aiforfinancialinformation)"
        )
    }
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # Remove noisy tags
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        return soup.get_text(separator="\n", strip=True)
    except Exception as e:
        print(f"  WARNING: Could not scrape {url}\n  Reason: {e}")
        return ""

scraped_texts = {}
for source in WEB_SOURCES:
    print(f"Scraping: {source['name']} ...")
    text = scrape_url(source["url"])
    scraped_texts[source["name"]] = text
    if text:
        print(f"  -> {len(text):,} characters")
    else:
        print("  -> (skipped — no content)")

scraped_count = sum(1 for t in scraped_texts.values() if t)
print(f"\nSuccessfully scraped {scraped_count}/{len(WEB_SOURCES)} web source(s).")

Scraping: NOLO – Illegal Lease and Rental Clauses ...


  Reason: 404 Client Error: Not Found for url: https://www.nolo.com/legal-encyclopedia/illegal-lease-rental-clauses.html
  -> (skipped — no content)
Scraping: HUD – Tenant Rights ...


  -> 241 characters
Scraping: Illinois Attorney General – Landlord Tenant Rights ...


  Reason: 404 Client Error:  for url: https://illinoisattorneygeneral.gov/rights/landlordtenants.html
  -> (skipped — no content)

Successfully scraped 1/3 web source(s).


## Cell 4: Combine Sources

In [4]:
# Combine all source text, truncating each to avoid exceeding token limits
MAX_CHARS_PER_PDF = 6000
MAX_CHARS_PER_WEB = 4000

combined_text = ""

for name, text in pdf_texts.items():
    combined_text += f"\n\n{'='*60}\nSOURCE (PDF): {name}\n{'='*60}\n"
    combined_text += text[:MAX_CHARS_PER_PDF]
    if len(text) > MAX_CHARS_PER_PDF:
        combined_text += f"\n... [truncated, {len(text):,} chars total]"

for name, text in scraped_texts.items():
    if not text:
        continue
    combined_text += f"\n\n{'='*60}\nSOURCE (Web): {name}\n{'='*60}\n"
    combined_text += text[:MAX_CHARS_PER_WEB]
    if len(text) > MAX_CHARS_PER_WEB:
        combined_text += f"\n... [truncated, {len(text):,} chars total]"

print(f"Combined source text: {len(combined_text):,} characters")
print(f"(PDFs: {len(pdf_texts)}, Web: {scraped_count})")

Combined source text: 10,312 characters
(PDFs: 2, Web: 1)


## Cell 5: LLM Extraction — Build Ground Truth

Ask GPT to extract a structured ground truth from the combined sources.

In [5]:
EXTRACTION_PROMPT = """You are a legal expert specializing in residential lease agreements and tenant rights law.

Based on the reference materials below, create a comprehensive ground truth document listing the types of 
lease clauses that a lease-review AI should be able to identify and classify.

For each clause type, provide:
- clause_category: a snake_case identifier (e.g., "habitability_waiver")
- name: a human-readable name
- expected_label: one of "illegal", "unfair_but_legal", "needs_negotiation", "fair"
  ("illegal" = clearly violates law; "unfair_but_legal" = legal but disadvantages tenant; 
   "needs_negotiation" = standard but worth pushing back on; "fair" = reasonable and balanced)
- severity_range: [min, max] integer on 1-10 scale (10 = most harmful to tenant)
- key_indicators: list of 2-4 language patterns that signal this clause type
- what_to_flag: one sentence describing what the agent should identify
- description: one sentence explaining why this clause is noteworthy

Include 15-20 diverse clause types covering: illegal clauses, unfair-but-legal clauses, 
negotiable clauses, and at least 2 examples of fair/balanced clauses.
Focus on residential leases in Illinois/Chicago but include generally applicable types.

Return ONLY valid JSON in this exact format:
{"ground_truth_clauses": [ ... ]}

Reference materials:
"""

print("Calling GPT to extract ground truth... (this may take 20-40 seconds)")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": EXTRACTION_PROMPT + combined_text,
        }
    ],
    response_format={"type": "json_object"},
    temperature=0.2,
)

raw_result = json.loads(response.choices[0].message.content)
clauses = raw_result.get("ground_truth_clauses", [])

print(f"Extracted {len(clauses)} ground truth clause types.")
print(f"Tokens used: {response.usage.total_tokens:,}")

Calling GPT to extract ground truth... (this may take 20-40 seconds)


Extracted 17 ground truth clause types.
Tokens used: 4,456


## Cell 6: Save Ground Truth

In [6]:
ground_truth = {
    "version": "1.0",
    "created": str(date.today()),
    "description": "Ground truth for evaluating AI lease agreement analysis output.",
    "sources": {
        "pdfs": list(pdf_texts.keys()),
        "web": [name for name, text in scraped_texts.items() if text],
    },
    "label_schema": {
        "illegal": "Clearly violates applicable law; unenforceable",
        "unfair_but_legal": "Legal but significantly disadvantages the tenant",
        "needs_negotiation": "Standard but tenant should push back or clarify",
        "fair": "Reasonable and balanced; acceptable as-is",
    },
    "severity_scale": "1 (minor inconvenience) to 10 (severe legal/financial risk to tenant)",
    "clause_types": clauses,
}

output_path = OUTPUT_DIR / "ground_truth.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print(f"Ground truth saved to: {output_path.resolve()}")
print(f"Total clause types: {len(clauses)}")

Ground truth saved to: C:\Users\fblel\OneDrive\Desktop\Courses\Winter 2026\AI Financial Info\Files\busn30135_labs\aiforfinancialinformation\evaluation\ground_truth\ground_truth.json
Total clause types: 17


## Cell 7: Preview Ground Truth

In [7]:
import pandas as pd

# Build a summary table
rows = []
for c in clauses:
    sev = c.get("severity_range", ["?", "?"])
    rows.append({
        "#": len(rows) + 1,
        "Category": c.get("clause_category", ""),
        "Name": c.get("name", ""),
        "Expected Label": c.get("expected_label", ""),
        "Severity Range": f"{sev[0]}–{sev[1]}",
    })

df_gt = pd.DataFrame(rows).set_index("#")
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 25)
print("Ground Truth Clause Types:")
print(df_gt.to_string())

# Label distribution
print("\nLabel distribution:")
print(df_gt["Expected Label"].value_counts().to_string())

Ground Truth Clause Types:
                          Category                                     Name     Expected Label Severity Range
#                                                                                                            
1              habitability_waiver            Waiver of Habitability Rights            illegal          10–10
2     eviction_without_due_process        Eviction Without Judicial Process            illegal           9–10
3             acceleration_of_rent            Acceleration of Rent Payments            illegal            8–9
4               attorney_fee_shift     Tenant Pays Landlord's Attorney Fees            illegal            7–8
5           confession_of_judgment            Confession of Judgment Clause            illegal           9–10
6              liability_exemption             Landlord Liability Exemption            illegal            8–9
7    security_deposit_restrictions  Excessive Security Deposit Requirements            illega